# 2025년에 학습한 모델로 2026년 예측하기

기존 피처 실험과 분리한 새 출발이다. **모델 선택은 2025년, 평가만 2026년**으로 구분한다.

1. 2025년 경기에서만 3개 시간순 구간으로 후보 60개를 비교한다. 각 구간의 과거 내부 validation에서만 early stopping한다.
2. 2025년 평가 정확도가 가장 높은 후보를 선택한다. 동률이면 log loss, 입력 수 순으로 비교한다.
3. 선택 모델의 seed별 2025년 최적 epoch 중앙값을 고정하고, 2025년 209시리즈 전체로 재학습한다. scaler도 2025년에만 fit한다.
4. 모델과 scaler를 저장한 다음 2026년 데이터를 읽는다. 2026년에는 학습하지 않는다.
5. 시리즈 시작 직전까지의 최근 경기 기록으로 입력을 갱신한다. 과거 2026년 경기 기록을 입력에 쓰는 것은 재학습과 다르다.

**해석의 한계:** 이전 실험에서 이미 2026년을 봤으므로 독립적인 미관측 test라고 부르지 않는다. 새 실행 안에서는 2026년 결과로 피처나 epoch를 선택하지 않는다. Bo3/Bo5는 완료된 시리즈 승수로 복원했으며, 실제 운영에서는 시작 전 대회 형식이 알려져 있다는 가정이다.

원본과 기존 실험 기록, 서비스 모델은 덮어쓰지 않는다. 새 결과는 `experiments/08_year_holdout`에 저장한다.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

ROOT = Path('/Users/seungyunmok/Developer/LOL_ML')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RESULTS = ROOT / 'notebook/experiments/08_year_holdout'


## 전체 실험을 다시 실행하려면

아래 변수를 True로 바꾸면 데이터 준비부터 60개 후보 학습, 모델 고정, 2026년 평가까지 다시 실행한다. False이면 저장된 결과를 읽는다. 구현은 `backend/training/year_holdout.py`에 모아 놓았다. 실행 중 2026년 결과를 보고 설정을 변경하지 않는다.


In [2]:
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    from backend.training.year_holdout import run
    run()


## 1. 2025년에서 무엇을 골랐는가?

`selection_metrics`는 후보를 고르는 데 사용한 개발 성능이다. 최종 2026년 성능과 구분한다. 초기에 학습용으로 필요한 경기를 제외한 159시리즈를 2025년 내 시간순 검증에 사용했다.


In [3]:
selection = json.loads((RESULTS / 'selection_lock.json').read_text())
print(json.dumps(selection, ensure_ascii=False, indent=2))
display(pd.read_csv(RESULTS / 'folds_2025.csv'))
display(pd.read_csv(RESULTS / 'selection_2025.csv').head(10))


{
  "variant": "candidate_08",
  "columns": [
    "diff_strong_lead15_rate",
    "is_bo5"
  ],
  "epochs": {
    "42": 1,
    "43": 38,
    "44": 1
  },
  "train_series": 209,
  "train_start": "2025-01-15 08:09:30",
  "train_end": "2025-09-28 05:30:31",
  "selection_metrics": {
    "variant": "candidate_08",
    "input_count": 2,
    "n": 159,
    "accuracy": 0.6477987421383647,
    "roc_auc": 0.6308203991130821,
    "log_loss": 0.6897890886531216,
    "brier_score": 0.24832374348448763,
    "probability_min": 0.4052920341491699,
    "probability_max": 0.5947079956531525
  }
}


,fold,fit,early_stop,future,future_start,future_end
0,1,38,12,60,2025-04-10 08:06:04,2025-05-21 11:23:37
1,2,86,24,54,2025-05-22 08:07:11,2025-08-10 08:19:07
2,3,131,33,45,2025-08-13 08:10:10,2025-09-28 05:30:31


,variant,input_count,n,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,candidate_08,2,159,0.647799,0.630820,0.689789,0.248324,0.405292,0.594708
1,candidate_23,3,159,0.641509,0.639531,0.672642,0.240034,0.314323,0.746231
2,candidate_02,2,159,0.641509,0.644124,0.685159,0.246014,0.439320,0.571886
3,candidate_17,3,159,0.635220,0.645787,0.670212,0.239258,0.183104,0.840944
4,candidate_21,3,159,0.635220,0.626703,0.681727,0.244350,0.393854,0.647972
5,candidate_44,3,159,0.635220,0.621872,0.691040,0.247585,0.126031,0.767898
6,candidate_26,3,159,0.628931,0.644441,0.674272,0.241048,0.226498,0.838621
7,candidate_42,3,159,0.628931,0.606747,0.703481,0.252399,0.118765,0.853630
8,candidate_18,3,159,0.622642,0.634305,0.667308,0.238219,0.204892,0.844777
9,candidate_47,3,159,0.622642,0.633196,0.685264,0.245707,0.226811,0.759132


## 2. 모델을 고정하고 2026년을 예측한 결과

정확도는 세 seed 모델의 확률을 평균해 한 번 예측한 결과다. 개별 정확도의 평균이 아니다. 비교용 로지스틱 회귀도 선택된 동일 입력과 2025년 학습 데이터만 사용한다. 결과를 본 뒤 선택 모델을 바꾸지 않는다.


In [4]:
evaluation = pd.read_csv(RESULTS / 'evaluation_2026.csv')
display(evaluation.round(4))

,model,n,correct,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,selected_mlp,186,109,0.5860,0.6360,0.6886,0.2477,0.4748,0.5180
1,linear_same_features,186,109,0.5860,0.6351,0.6661,0.2367,0.2348,0.6702
2,always_50,186,92,0.4946,0.5000,0.6931,0.2500,0.5000,0.5000
3,train_winrate,186,94,0.5054,0.5000,0.6936,0.2502,0.4785,0.4785


## 3. 월별 결과

월별 표본 수가 다르므로 정확도만 비교하지 않는다. 특히 3월 1시리즈의 100%는 성능 우위의 근거가 아니다. 2026년 데이터는 9월 12일까지다.


In [5]:
monthly = pd.read_csv(RESULTS / 'monthly_2026.csv')
display(monthly.round(4))

,month,n,correct,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,2026-01,24,14,0.5833,0.6926,0.6864,0.2466,0.4857,0.5143
1,2026-02,15,9,0.6000,0.1364,0.6981,0.2525,0.4857,0.5180
2,2026-03,1,1,1.0000,NaN,0.6893,0.2481,0.4981,0.4981
3,2026-04,44,28,0.6364,0.7372,0.6835,0.2452,0.4748,0.5143
4,2026-05,46,28,0.6087,0.6875,0.6863,0.2466,0.4820,0.5143
5,2026-06,5,3,0.6000,0.7500,0.6873,0.2471,0.5000,0.5108
6,2026-07,6,4,0.6667,0.6667,0.6877,0.2473,0.4857,0.5085
7,2026-08,38,18,0.4737,0.4855,0.6944,0.2506,0.4857,0.5180
8,2026-09,7,4,0.5714,0.3500,0.6939,0.2504,0.4981,0.5108


## 4. 해석

- 선택된 입력: 최근 5세트의 **15분 1,000골드 이상 리드율 차이**와 **Bo5 여부**, 총 2개.
- 2026년 **109/186 정답 = 58.60%**, log loss **0.6886**, ROC-AUC **0.6360**.
- 같은 입력의 로지스틱 회귀도 **58.60%**이고 log loss는 **0.6661**로 더 낮다. 딥러닝의 우위는 확인되지 않았다.
- MLP 확률 범위는 약 **47.48~51.80%**로 좁다. 정답 방향을 고르는 능력과 확률의 유용성은 구분해야 한다.
- 2025년 내부 검증이 정한 최종 epoch는 seed별 **1, 38, 1**이다. 두 모델은 학습 초기에 선택됐다. 2026년 결과가 약하다고 이번 평가 뒤에 epoch를 늘리면 새 실험이 된다.
- 기존의 70.51%와 평가 대상·학습 기간이 다르므로 성능이 개선/악화됐다고 직접 비교하지 않는다.
- 모델은 실험용으로 저장했고 서비스에 배포하지 않았다.


In [6]:
predictions = pd.read_csv(RESULTS / 'predictions_2026.csv')
display(predictions.head(10))
print(json.dumps(json.loads((RESULTS / 'manifest.json').read_text()), indent=2))

,series_key,date,team1,team2,best_of,result,probability,p_seed_42,p_seed_43,p_seed_44,correct
0,2026|Cup|2026-01-14|DN SOOPers|KT Rolster,2026-01-14 08:13:43,DN SOOPers,KT Rolster,3,0,0.508513,0.504113,0.523058,0.498369,False
1,2026|Cup|2026-01-14|Dplus Kia|HANJIN BRION,2026-01-14 11:22:18,Dplus Kia,HANJIN BRION,3,1,0.514252,0.510700,0.536037,0.496018,True
2,2026|Cup|2026-01-15|Gen.G|Kiwoom DRX,2026-01-15 08:05:23,Gen.G,Kiwoom DRX,3,1,0.491487,0.495887,0.476942,0.501631,False
3,2026|Cup|2026-01-15|BNK FEARX|Nongshim RedForce,2026-01-15 10:15:24,BNK FEARX,Nongshim RedForce,3,1,0.491487,0.495887,0.476942,0.501631,False
4,2026|Cup|2026-01-16|DN SOOPers|Dplus Kia,2026-01-16 08:05:40,DN SOOPers,Dplus Kia,3,0,0.485748,0.489300,0.463963,0.503981,True
5,2026|Cup|2026-01-16|Hanwha Life Esports|T1,2026-01-16 10:08:16,Hanwha Life Esports,T1,3,0,0.500000,0.500000,0.500000,0.500000,False
6,2026|Cup|2026-01-17|BNK FEARX|HANJIN BRION,2026-01-17 08:05:53,BNK FEARX,HANJIN BRION,3,1,0.508513,0.504113,0.523058,0.498369,True
7,2026|Cup|2026-01-17|Gen.G|KT Rolster,2026-01-17 11:10:45,Gen.G,KT Rolster,3,1,0.500000,0.500000,0.500000,0.500000,True
8,2026|Cup|2026-01-18|Hanwha Life Esports|Nongsh...,2026-01-18 08:05:24,Hanwha Life Esports,Nongshim RedForce,3,0,0.500000,0.500000,0.500000,0.500000,False
9,2026|Cup|2026-01-18|Kiwoom DRX|T1,2026-01-18 10:20:18,Kiwoom DRX,T1,3,0,0.485748,0.489300,0.463963,0.503981,True


{
  "checkpoint_sha256": "51ef3e1d60e4ec1e0f5d4f8d69f6f70e75b83836f1ce6a447f5fcc804ee03d15",
  "source_sha256": {
    "2025": "c9a158b9e0a965a47d31d3674c127a26f75e6c91a324bd1858e4784b1336214a",
    "2026": "024330eb7a03e07c1aba55e17abf2e55f8e79e46e42e3f47389194173e59730a"
  },
  "checks": [
    "2025 feature prefix equality",
    "completed prefix invariance",
    "frozen checkpoint unchanged"
  ],
  "test_start": "2026-01-14 08:13:43",
  "test_end": "2026-09-12 05:21:16"
}
